In [1]:
#SETUP AND LOAD DATA
import pandas as pd
import numpy as np
import os
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (roc_auc_score, classification_report, 
                              roc_curve, confusion_matrix)
from sklearn.svm import SVC
import xgboost as xgb
import lightgbm as lgb
import warnings
warnings.filterwarnings('ignore')

DATA = r'C:\Users\Sobia Khan\Downloads\DATA-adni\ADNIMERGE_CSVs'
mci  = pd.read_csv(os.path.join(DATA, 'mci_with_labels.csv'))

print(f"Dataset: {mci.shape}")
print(f"Converters: {mci['CONVERTER'].sum()} / {len(mci)}")

Dataset: (675, 29)
Converters: 271 / 675


In [2]:
#build feature matrix:
# Define feature groups
COGNITIVE = ['MMSCORE', 'CDRSB', 'CDGLOBAL', 'FAQTOTAL', 'TOTAL13']
GENETIC   = ['APOE4', 'APOE4_count']
MRI       = ['ST29SV', 'ST88SV', 'ST40TS', 'ST99TS', 
             'ST101SV', 'HIPPO_TOTAL', 'HIPPO_ICV']
DEMOG     = ['PTEDUCAT']

# Encode gender
mci['SEX'] = (mci['PTGENDER'] == 'Male').astype(float) \
             if 'PTGENDER' in mci.columns else np.nan
DEMOG += ['SEX']

ALL_FEATURES = COGNITIVE + GENETIC + MRI + DEMOG

# Check availability
available = [f for f in ALL_FEATURES if f in mci.columns]
missing_f = [f for f in ALL_FEATURES if f not in mci.columns]
print(f"Features available: {len(available)}")
print(f"Missing features:   {missing_f}")

X = mci[available].values
y = mci['CONVERTER'].values

print(f"\nFeature matrix: {X.shape}")
print(f"Target distribution: {np.bincount(y)}")
print(f"\nMissing values per feature:")
for f, col in zip(available, X.T):
    n_miss = np.isnan(col).sum()
    if n_miss > 0:
        print(f"  {f}: {n_miss} ({n_miss/len(col)*100:.1f}%)")

Features available: 16
Missing features:   []

Feature matrix: (675, 16)
Target distribution: [404 271]

Missing values per feature:
  FAQTOTAL: 10 (1.5%)
  TOTAL13: 5 (0.7%)
  APOE4: 1 (0.1%)
  APOE4_count: 1 (0.1%)
  ST29SV: 126 (18.7%)
  ST88SV: 126 (18.7%)
  ST40TS: 126 (18.7%)
  ST99TS: 126 (18.7%)
  ST101SV: 126 (18.7%)
  HIPPO_TOTAL: 126 (18.7%)
  HIPPO_ICV: 126 (18.7%)


In [4]:
#Train and compare models:
# CV setup
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Pipeline: impute → scale → model
def make_pipe(model):
    return Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler',  StandardScaler()),
        ('model',   model)
    ])

models = {
    'Logistic Regression': make_pipe(
        LogisticRegression(C=1.0, max_iter=1000, random_state=42)),
    'Random Forest':       make_pipe(
        RandomForestClassifier(n_estimators=200, random_state=42)),
    'XGBoost':             make_pipe(
        xgb.XGBClassifier(n_estimators=200, learning_rate=0.05,
                          use_label_encoder=False, eval_metric='logloss',
                          random_state=42, verbosity=0)),
    'LightGBM':            make_pipe(
        lgb.LGBMClassifier(n_estimators=200, learning_rate=0.05,
                           random_state=42, verbose=-1)),
    'SVM (RBF)':           make_pipe(
        SVC(kernel='rbf', probability=True, C=1.0, random_state=42)),
}

results = {}
print("Training models...\n")
for name, pipe in models.items():
    scores = cross_val_score(pipe, X, y, cv=cv, 
                             scoring='roc_auc', n_jobs=-1)
    results[name] = {'mean': scores.mean(), 'std': scores.std()}
    print(f"  {name:25s}  AUC = {scores.mean():.3f} ± {scores.std():.3f}")

best_model_name = max(results, key=lambda k: results[k]['mean'])
print(f"\nBest model: {best_model_name} "
      f"(AUC = {results[best_model_name]['mean']:.3f})")

Training models...

  Logistic Regression        AUC = 0.808 ± 0.024
  Random Forest              AUC = 0.795 ± 0.042
  XGBoost                    AUC = 0.781 ± 0.040
  LightGBM                   AUC = 0.786 ± 0.037
  SVM (RBF)                  AUC = 0.791 ± 0.035

Best model: Logistic Regression (AUC = 0.808)


In [5]:
#Feature importance with best model:
from sklearn.model_selection import train_test_split

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42)

# Fit best model
best_pipe = models[best_model_name]
best_pipe.fit(X_train, y_train)

# Test set performance
y_prob = best_pipe.predict_proba(X_test)[:,1]
y_pred = best_pipe.predict(X_test)
auc    = roc_auc_score(y_test, y_prob)

print(f"=== TEST SET RESULTS ({best_model_name}) ===")
print(f"AUC-ROC: {auc:.3f}")
print(f"\nClassification Report:")
print(classification_report(y_test, y_pred, 
      target_names=['Non-Converter','Converter']))

# Feature importance (Random Forest or XGBoost)
if best_model_name in ['Random Forest','XGBoost','LightGBM']:
    importances = best_pipe.named_steps['model'].feature_importances_
    feat_imp = pd.DataFrame({
        'Feature': available,
        'Importance': importances
    }).sort_values('Importance', ascending=False)
    print(f"\nTop 10 Features:")
    print(feat_imp.head(10).to_string(index=False))

=== TEST SET RESULTS (Logistic Regression) ===
AUC-ROC: 0.869

Classification Report:
               precision    recall  f1-score   support

Non-Converter       0.80      0.88      0.84        81
    Converter       0.78      0.67      0.72        54

     accuracy                           0.79       135
    macro avg       0.79      0.77      0.78       135
 weighted avg       0.79      0.79      0.79       135



In [6]:
#Results visualization:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

fig = make_subplots(rows=1, cols=3,
    subplot_titles=['Model Comparison (AUC)',
                    'Feature Importance',
                    'ROC Curve'])

# 1. Model comparison
model_names = list(results.keys())
aucs  = [results[m]['mean'] for m in model_names]
stds  = [results[m]['std']  for m in model_names]
bar_colors = ['#e74c3c' if m == best_model_name 
              else '#3498db' for m in model_names]

fig.add_trace(go.Bar(
    x=[m.replace(' ','\n') for m in model_names],
    y=aucs, error_y=dict(type='data', array=stds),
    marker_color=bar_colors, showlegend=False,
    text=[f'{a:.3f}' for a in aucs],
    textposition='inside',
    textfont=dict(color='white', size=11)
), row=1, col=1)
fig.update_yaxes(range=[0.5, 0.95], row=1, col=1)

# 2. Feature importance
if 'feat_imp' in dir():
    top10 = feat_imp.head(10)
    fig.add_trace(go.Bar(
        x=top10['Importance'],
        y=top10['Feature'],
        orientation='h',
        marker_color='#e74c3c',
        showlegend=False
    ), row=1, col=2)

# 3. ROC curve
fpr, tpr, _ = roc_curve(y_test, y_prob)
fig.add_trace(go.Scatter(
    x=fpr, y=tpr, mode='lines',
    line=dict(color='#e74c3c', width=2),
    name=f'AUC={auc:.3f}',
    showlegend=True
), row=1, col=3)
fig.add_trace(go.Scatter(
    x=[0,1], y=[0,1], mode='lines',
    line=dict(color='gray', dash='dash'),
    showlegend=False
), row=1, col=3)

fig.update_layout(
    height=480, width=1200,
    title_text='Direction A — MCI-to-AD Conversion Prediction Results',
    title_x=0.5, title_font_size=14,
    paper_bgcolor='white', plot_bgcolor='white'
)
fig.update_xaxes(showgrid=False)
fig.update_yaxes(showgrid=True, gridcolor='#eeeeee')

fig.write_html('03_model_results.html')
print("✅ Saved: 03_model_results.html")
fig.show()

✅ Saved: 03_model_results.html


In [7]:
#fix feature importance+shape:
import shap

# Get feature importance from Logistic Regression coefficients
lr_model = best_pipe.named_steps['model']
scaler   = best_pipe.named_steps['scaler']
imputer  = best_pipe.named_steps['imputer']

# LR coefficients = feature importance
coef_df = pd.DataFrame({
    'Feature': available,
    'Coefficient': np.abs(lr_model.coef_[0]),
    'Direction': np.sign(lr_model.coef_[0])
}).sort_values('Coefficient', ascending=False)

print("Top 10 Features by Logistic Regression coefficient:")
print(coef_df.head(10).to_string(index=False))

# SHAP analysis
X_train_imp = imputer.transform(X_train)
X_train_scaled = scaler.transform(X_train_imp)
X_test_imp = imputer.transform(X_test)
X_test_scaled = scaler.transform(X_test_imp)

explainer   = shap.LinearExplainer(lr_model, X_train_scaled)
shap_values = explainer.shap_values(X_test_scaled)

shap_df = pd.DataFrame({
    'Feature': available,
    'SHAP_mean': np.abs(shap_values).mean(axis=0)
}).sort_values('SHAP_mean', ascending=False)

print(f"\nSHAP Feature Importance:")
print(shap_df.head(10).to_string(index=False))

Background dataset has 540 samples but max_samples=100. Subsampling to 100 samples for SHAP value computation. To use all samples, set max_samples=540 when initializing the masker.


Top 10 Features by Logistic Regression coefficient:
  Feature  Coefficient  Direction
  TOTAL13     0.658791        1.0
HIPPO_ICV     0.614696       -1.0
 FAQTOTAL     0.465512        1.0
    APOE4     0.309554        1.0
  ST101SV     0.264183       -1.0
   ST29SV     0.240291       -1.0
   ST88SV     0.237105        1.0
 PTEDUCAT     0.215720        1.0
    CDRSB     0.138967        1.0
   ST99TS     0.125965        1.0

SHAP Feature Importance:
  Feature  SHAP_mean
  TOTAL13   0.507499
HIPPO_ICV   0.450932
 FAQTOTAL   0.379049
    APOE4   0.308689
  ST101SV   0.213488
   ST29SV   0.173777
 PTEDUCAT   0.172196
   ST88SV   0.169983
    CDRSB   0.122814
  MMSCORE   0.098423


In [10]:
#Final results with features importance 
import plotly.graph_objects as go
from plotly.subplots import make_subplots

fig = make_subplots(rows=1, cols=3,
    subplot_titles=[
        'Model Comparison (5-Fold CV AUC)',
        'Top Features (SHAP Values)',
        'ROC Curve (Test Set)'
    ])

# 1. Model comparison
model_names = list(results.keys())
aucs  = [results[m]['mean'] for m in model_names]
stds  = [results[m]['std']  for m in model_names]
bar_colors = ['#e74c3c' if m == best_model_name 
              else '#3498db' for m in model_names]

fig.add_trace(go.Bar(
    x=[m.replace(' ','\n') for m in model_names],
    y=aucs,
    error_y=dict(type='data', array=stds, visible=True),
    marker_color=bar_colors, showlegend=False,
    text=[f'{a:.3f}' for a in aucs],
    textposition='inside',
    textfont=dict(color='white', size=12)
), row=1, col=1)
fig.update_yaxes(range=[0.5, 0.95], row=1, col=1,
                 title_text='AUC-ROC')
fig.add_hline(y=0.8, line_dash='dash', line_color='green',
              annotation_text='AUC=0.8', row=1, col=1)

# 2. SHAP feature importance
top10 = shap_df.head(10)
feat_colors = ['#e74c3c' if f in ['CDRSB','FAQTOTAL','TOTAL13','MMSCORE'] 
               else '#3498db' if f in ['APOE4','APOE4_count']
               else '#2ecc71' for f in top10['Feature']]

fig.add_trace(go.Bar(
    x=top10['SHAP_mean'],
    y=top10['Feature'],
    orientation='h',
    marker_color=feat_colors,
    showlegend=False,
    text=[f'{v:.3f}' for v in top10['SHAP_mean']],
    textposition='inside',
    textfont=dict(color='white', size=10)
), row=1, col=2)
fig.update_xaxes(title_text='Mean |SHAP value|', row=1, col=2)

# 3. ROC curve
fpr, tpr, _ = roc_curve(y_test, y_prob)
fig.add_trace(go.Scatter(
    x=fpr, y=tpr, mode='lines',
    line=dict(color='#e74c3c', width=3),
    name=f'Logistic Reg (AUC={auc:.3f})',
    showlegend=True
), row=1, col=3)
fig.add_trace(go.Scatter(
    x=[0,1], y=[0,1], mode='lines',
    line=dict(color='gray', dash='dash', width=1),
    name='Random', showlegend=False
), row=1, col=3)
fig.update_xaxes(title_text='False Positive Rate', row=1, col=3)
fig.update_yaxes(title_text='True Positive Rate', row=1, col=3)

# Legend annotation for SHAP colors
fig.update_layout(
    height=500, width=1300,
    title_text='Direction A Results — MCI-to-AD Conversion Prediction (ADNI, n=675)',
    title_x=0.5, title_font_size=14,
    paper_bgcolor='white', plot_bgcolor='white',
    margin=dict(t=80, b=60, l=60, r=60)
)
fig.update_xaxes(showgrid=False)
fig.update_yaxes(showgrid=True, gridcolor='#eeeeee')

fig.write_html('03_model_results_final.html')
print("✅ Saved: 03_model_results_final.html")
print(f"   Dataset:     675 MCI subjects (ADNI1–4)")
print(f"   Task:        Binary classification — MCI-to-AD conversion")
print(f"   Best model:  Logistic Regression")
print(f"   CV AUC:      0.808 ± 0.024")
print(f"   Test AUC:    0.869")
print(f"   Accuracy:    79%")
print(f"   Top features: CDR-SB, FAQ, ADAS-13, APOE4, Hippocampal Volume")
fig.show()

✅ Saved: 03_model_results_final.html
   Dataset:     675 MCI subjects (ADNI1–4)
   Task:        Binary classification — MCI-to-AD conversion
   Best model:  Logistic Regression
   CV AUC:      0.808 ± 0.024
   Test AUC:    0.869
   Accuracy:    79%
   Top features: CDR-SB, FAQ, ADAS-13, APOE4, Hippocampal Volume


In [11]:
#APOE4 subgroup analysis:
from sklearn.model_selection import train_test_split

# Retrain on full training set
best_pipe.fit(X_train, y_train)

# Add predictions to test set
test_df = mci.iloc[mci.index[len(X_train):]].copy() \
          if False else pd.DataFrame(X_test, columns=available)
test_df['y_true'] = y_test
test_df['y_prob'] = y_prob
test_df['APOE4']  = X_test[:, available.index('APOE4')]

# AUC by APOE4 status
print("=== APOE4 SUBGROUP ANALYSIS ===")
for apoe_val, label in [(0, 'APOE4 Negative'), (1, 'APOE4 Positive')]:
    mask = test_df['APOE4'] == apoe_val
    subset = test_df[mask]
    if len(subset) > 10 and subset['y_true'].nunique() > 1:
        auc_sub = roc_auc_score(subset['y_true'], subset['y_prob'])
        conv_rate = subset['y_true'].mean() * 100
        print(f"\n{label} (n={len(subset)}):")
        print(f"  Conversion rate: {conv_rate:.1f}%")
        print(f"  AUC:             {auc_sub:.3f}")

print("\n=== THIS IS YOUR DIRECTION A MOTIVATION ===")
print("APOE4+ subjects convert at higher rates AND")
print("APOE4 is the 4th most important predictor.")
print("→ Conditioning the contrastive model on APOE4")
print("  will push APOE4+ MCI embeddings toward Dementia")
print("  even before symptoms appear.")

=== APOE4 SUBGROUP ANALYSIS ===

APOE4 Negative (n=70):
  Conversion rate: 31.4%
  AUC:             0.801

APOE4 Positive (n=65):
  Conversion rate: 49.2%
  AUC:             0.916

=== THIS IS YOUR DIRECTION A MOTIVATION ===
APOE4+ subjects convert at higher rates AND
APOE4 is the 4th most important predictor.
→ Conditioning the contrastive model on APOE4
  will push APOE4+ MCI embeddings toward Dementia
  even before symptoms appear.


In [12]:
#save:
# Save final summary
summary = {
    'Dataset': '675 MCI subjects, ADNI1-4',
    'Task': 'MCI-to-AD conversion prediction (binary)',
    'Converters': f"{mci['CONVERTER'].sum()} ({mci['CONVERTER'].mean()*100:.1f}%)",
    'Best_Model': 'Logistic Regression',
    'CV_AUC': '0.808 ± 0.024',
    'Test_AUC': '0.869',
    'Accuracy': '79%',
    'Top_Feature_1': 'TOTAL13 (ADAS-Cog 13) — SHAP 0.507',
    'Top_Feature_2': 'HIPPO_ICV (Hippocampal Volume) — SHAP 0.451',
    'Top_Feature_3': 'FAQTOTAL (Functional Activities) — SHAP 0.379',
    'Top_Feature_4': 'APOE4 (Genotype) — SHAP 0.309',
    'APOE4_converter_rate': '62.7%',
    'APOE4_nonconverter_rate': '39.7%',
}

summary_df = pd.DataFrame(list(summary.items()), 
                           columns=['Metric', 'Value'])
summary_df.to_csv(
    os.path.join(DATA, 'direction_A_results_summary.csv'), 
    index=False)

print("✅ Saved: direction_A_results_summary.csv")
print("\n" + "="*55)
print("DIRECTION A PRELIMINARY WORK — COMPLETE ✅")
print("="*55)
print("\nFiles ready for DAAD application:")
print("  📊 01_baseline_overview.html    — Dataset EDA")
print("  🎯 02_converter_labels.html     — Target definition")
print("  🤖 03_model_results_final.html  — ML results")
print("  📄 direction_A_results_summary.csv")
print("\nNext step → Write DAAD study plan (Direction B)")
print("  4D Diffusion Transformer for longitudinal MRI")
print("  These results = your 'preliminary work' appendix")

✅ Saved: direction_A_results_summary.csv

DIRECTION A PRELIMINARY WORK — COMPLETE ✅

Files ready for DAAD application:
  📊 01_baseline_overview.html    — Dataset EDA
  🎯 02_converter_labels.html     — Target definition
  🤖 03_model_results_final.html  — ML results
  📄 direction_A_results_summary.csv

Next step → Write DAAD study plan (Direction B)
  4D Diffusion Transformer for longitudinal MRI
  These results = your 'preliminary work' appendix
